# Microdosing Outcomes

Analysis of treatment sentiment reports from r/microdosing (1-month snapshot, March–April 2026).  
936 sentiment reports · 243 unique users · 167 substances.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
from pathlib import Path

# Resolve project root regardless of working directory
_here = Path('.').resolve()
PROJECT_ROOT = _here.parent if (_here / 'app').exists() is False and (_here.parent / 'app').exists() else _here
sys.path.insert(0, str(PROJECT_ROOT))

from app.analysis.stats import (
    get_user_sentiment,
    run_binomial_test,
    summarize_drug,
    run_comparison,
    run_kruskal_wallis,
    REPORTING_BIAS_DISCLAIMER,
)

DB_PATH = PROJECT_ROOT / 'microdosing.db'
conn = sqlite3.connect(str(DB_PATH))
print('Connected to', DB_PATH)

## 2. Data Exploration

Quick overview of what's in the database: row counts, date range, and top substances by number of unique reporters.

In [ ]:
import datetime

n_reports = conn.execute('SELECT COUNT(*) FROM treatment_reports').fetchone()[0]
n_users   = conn.execute('SELECT COUNT(DISTINCT user_id) FROM treatment_reports').fetchone()[0]
n_drugs   = conn.execute('SELECT COUNT(DISTINCT drug_id) FROM treatment_reports').fetchone()[0]
lo, hi    = conn.execute('SELECT MIN(post_date), MAX(post_date) FROM posts').fetchone()

print(f'Reports : {n_reports}')
print(f'Users   : {n_users}')
print(f'Drugs   : {n_drugs}')
print(f'Dates   : {datetime.datetime.fromtimestamp(lo).date()} → {datetime.datetime.fromtimestamp(hi).date()}')

# Top drugs by unique users (excluding generic "microdosing" label)
top_drugs_df = pd.read_sql("""
    SELECT t.canonical_name AS drug, COUNT(DISTINCT tr.user_id) AS n_users
    FROM treatment_reports tr
    JOIN treatment t ON t.id = tr.drug_id
    WHERE t.canonical_name != 'microdosing'
    GROUP BY t.canonical_name
    ORDER BY n_users DESC
    LIMIT 15
""", conn)

print('\nTop 15 substances by unique reporters:')
print(top_drugs_df.to_string(index=False))

## 3. Analysis

### 3a. Psilocybin — outcome summary and significance test

Psilocybin has the largest sample (166 users), making it the only substance with adequate power for a formal test. We use a one-sample binomial test against a baseline rate of 50% (i.e., is the positive outcome rate better than chance?).

In [ ]:
df_psilo = get_user_sentiment(conn, 'psilocybin')
summary  = summarize_drug(df_psilo, 'psilocybin')
binom    = run_binomial_test(df_psilo)

print(f'Psilocybin — n={summary.n_users} users ({summary.n_posts} posts)')
print(f'  Positive : {summary.pct_positive:.1f}%  (95% CI: {summary.pct_positive_ci[0]:.1f}–{summary.pct_positive_ci[1]:.1f}%)')
print(f'  Mixed    : {summary.pct_mixed:.1f}%')
print(f'  Neutral  : {summary.pct_neutral:.1f}%')
print(f'  Negative : {summary.pct_negative:.1f}%')
print(f'  Mean sentiment score: {summary.mean_sentiment:.3f}  (scale: −1 to +1)')
print(f'  Binomial test vs 50% baseline: p = {binom.p_value:.2e}')

for w in binom.warnings:
    print(f'  [{w.severity.upper()}] {w.message}')

### 3b. Psilocybin vs LSD — direct comparison

Both psilocybin (n=166) and LSD (n=30) have enough users for a basic comparison. We use a Mann-Whitney U test on continuous sentiment scores plus a chi-square test on outcome categories. Note the large imbalance in sample sizes.

In [ ]:
df_lsd  = get_user_sentiment(conn, 'lsd')
summary_lsd = summarize_drug(df_lsd, 'lsd')
comp    = run_comparison(df_psilo, df_lsd)

print(f'LSD — n={summary_lsd.n_users} users')
print(f'  Positive : {summary_lsd.pct_positive:.1f}%')
print(f'  Negative : {summary_lsd.pct_negative:.1f}%')
print(f'  Mean sentiment: {summary_lsd.mean_sentiment:.3f}')
print()
print(f'Mann-Whitney U: p = {comp.mw_p_value:.3f}  (effect r = {comp.mw_effect_size_r:.3f})')
print(f'Chi-square    : p = {comp.cat_p_value:.3f}  (effect = {comp.cat_effect_size:.3f})')
print(f'Significant   : {comp.mw_significant}')

for w in comp.warnings:
    print(f'[{w.severity.upper()}] {w.message}')

### 3c. Stamets Stack components

The Stamets Stack (psilocybin + lion's mane + niacin) is a popular microdosing protocol. We summarise each component's outcome rates independently.

In [ ]:
stack_drugs = ['psilocybin', "lion's mane", 'niacin', 'mushrooms']
stack_rows = []
for drug in stack_drugs:
    df = get_user_sentiment(conn, drug)
    if not df.empty:
        s = summarize_drug(df, drug)
        stack_rows.append({
            'Drug': drug, 'n': s.n_users,
            'Positive%': round(s.pct_positive, 1),
            'Mixed%': round(s.pct_mixed, 1),
            'Negative%': round(s.pct_negative, 1),
            'Mean score': round(s.mean_sentiment, 3),
        })

pd.DataFrame(stack_rows)

## 4. Visualization

### Methodology note

**How sentiment is scored.** The LLM reads each post or comment and assigns a sentiment label toward each substance mentioned: *positive* (1.0), *mixed* (0.5), *weak* (0.25), *neutral* (0.0), or *negative* (−1.0). These numeric scores are then averaged across all posts for a given user and substance to produce a single per-user mean score (*ū*).

**User-level aggregation.** Each data point represents one Reddit user's overall experience with a substance, not one post. A user who posted ten positive comments about psilocybin contributes a single data point (ū = 1.0), preventing prolific posters from skewing results.

**Classification thresholds** (symmetric):

| Category | Threshold |
|---|---|
| Positive | ū > 0.7 |
| Negative | ū < −0.7 |
| Mixed / Neutral | −0.7 ≤ ū ≤ 0.7 |

**What the chart shows.** Each bar represents one substance. Bars extend right (positive outcomes, green) and left (negative outcomes, red) from a central axis, with mixed/neutral outcomes (grey) spanning the centre. Substances are ranked by positive outcome rate. Only substances with ≥ 5 unique reporters are included.

In [ ]:
# Gather outcome data for all substances with >= 5 users, excluding 'microdosing'
eligible = pd.read_sql("""
    SELECT t.canonical_name AS drug, COUNT(DISTINCT tr.user_id) AS n
    FROM treatment_reports tr
    JOIN treatment t ON t.id = tr.drug_id
    WHERE t.canonical_name != 'microdosing'
    GROUP BY t.canonical_name
    HAVING n >= 5
    ORDER BY n DESC
    LIMIT 14
""", conn)

rows = []
for _, row in eligible.iterrows():
    df = get_user_sentiment(conn, row['drug'])
    if df.empty:
        continue
    s = summarize_drug(df, row['drug'])
    rows.append({
        'drug': row['drug'],
        'n': s.n_users,
        'pct_pos': s.pct_positive,
        'pct_mix': s.pct_mixed + s.pct_neutral,
        'pct_neg': s.pct_negative,
    })

chart_df = pd.DataFrame(rows).sort_values('pct_pos', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))

y = np.arange(len(chart_df))
pct_pos = chart_df['pct_pos'].values
pct_mix = chart_df['pct_mix'].values
pct_neg = chart_df['pct_neg'].values

ax.barh(y, pct_pos,  left=pct_mix/2,  color='#2ecc71', label='Positive')
ax.barh(y, -pct_neg, left=-pct_mix/2, color='#e74c3c', label='Negative')
ax.barh(y, pct_mix,  left=-pct_mix/2, color='#bdc3c7', label='Mixed / Neutral')

# n labels
for i, (_, row) in enumerate(chart_df.iterrows()):
    ax.text(pct_mix[i]/2 + pct_pos[i] + 1, i, f'n={row["n"]}',
            va='center', ha='left', fontsize=8, color='#555555')

ax.set_yticks(y)
ax.set_yticklabels([d.title() for d in chart_df['drug']])
ax.axvline(0, color='black', linewidth=0.8)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{abs(x):.0f}%'))
ax.set_xlabel('← Negative outcome rate  |  Positive outcome rate →')
ax.set_title('Microdosing Community: Treatment Outcome Distribution\n(r/microdosing, 1-month snapshot, March–April 2026)', pad=12)
ax.legend(loc='lower right', fontsize=9)
ax.set_xlim(-60, 105)
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'notebooks' / '3_microdosing_diverging.png'), dpi=150, bbox_inches='tight')
plt.show()

### Psilocybin vs LSD — outcome distribution comparison

Side-by-side stacked bar showing the proportion of users in each outcome category for the two most-reported psychedelic substances. The Mann-Whitney U test on continuous scores and chi-square test on categories are reported above (Section 3b). The chart is purely descriptive; formal significance is already reported.

In [ ]:
compare_drugs = {'Psilocybin': df_psilo, 'LSD': df_lsd}

fig, ax = plt.subplots(figsize=(6, 4))

categories = ['positive', 'mixed', 'neutral', 'negative']
colors     = ['#2ecc71', '#bdc3c7', '#ecf0f1', '#e74c3c']
bottoms    = np.zeros(len(compare_drugs))
x          = np.arange(len(compare_drugs))
labels     = list(compare_drugs.keys())

for cat, color in zip(categories, colors):
    vals = [
        (df['category'] == cat).sum() / len(df) * 100
        for df in compare_drugs.values()
    ]
    ax.bar(x, vals, bottom=bottoms, color=color, label=cat.title(), width=0.5)
    for xi, (v, b) in enumerate(zip(vals, bottoms)):
        if v > 5:
            ax.text(xi, b + v/2, f'{v:.0f}%', ha='center', va='center', fontsize=9, fontweight='bold')
    bottoms += np.array(vals)

# n labels and significance note
ns = [len(df) for df in compare_drugs.values()]
ax.set_xticks(x)
ax.set_xticklabels([f'{lbl}\n(n={n})' for lbl, n in zip(labels, ns)])
ax.set_ylabel('% of users')
ax.set_title('Psilocybin vs LSD — Outcome Distribution')

sig_text = f'p = {comp.mw_p_value:.2f} (Mann-Whitney U, not significant)'
ax.text(0.5, -0.18, sig_text, transform=ax.transAxes, ha='center', fontsize=8,
        color='gray', style='italic')

handles, lbls = ax.get_legend_handles_labels()
ax.legend(handles[::-1], lbls[::-1], loc='upper left', fontsize=8, bbox_to_anchor=(1.01, 1))
ax.set_ylim(0, 110)
plt.tight_layout()
plt.show()

## 5. Summary

**Psilocybin** is the dominant substance in this community (166 users, 70.5% positive outcomes, 95% CI 63–77%). The binomial test confirms this exceeds chance well beyond any reasonable significance threshold. The positive rate is among the highest seen in any PatientPunk cohort to date.

**LSD** shows a similar profile (76.7% positive, n=30) but the sample is too small and imbalanced to detect a meaningful difference from psilocybin. The Mann-Whitney U comparison is non-significant (p = 0.75), consistent with both being used in similar low-dose, mood-improving protocols by the same community.

**Stamets Stack components** (lion's mane n=15, niacin n=6) show high positive rates but n is too small for formal testing.

**SSRIs / antidepressants** (n=17) appear in this community likely in the context of tapering, interaction questions, or comparing to psychedelic effects — not as a primary treatment of interest. Sentiment is more mixed.

**Caveats:**
- 1-month snapshot only — behaviour patterns may differ across seasons or community events
- "microdosing" as a practice label was excluded from drug-level analysis
- No conditions data in this database — cannot stratify by indication
- r/microdosing is a community self-selected for positive interest in the practice

**Suggested next steps:** Run variable extraction on microdosing users to get condition/demographic data; compare psilocybin outcomes in depression vs anxiety vs ADHD reporters; add the 6-month r/microdosing dataset when available.

---

*Based on self-selected Reddit posts. Users who never posted about a treatment are not represented. Results reflect reporting patterns, not population-level treatment effects.*